# GPT Action 库：Sharepoint（以文档形式返回）

## 简介

本页面为开发者针对特定应用构建 GPT Action 提供说明和指南。在继续之前，请先熟悉以下内容： 
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零开始构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

该方案利用 Microsoft Graph API 的[搜索能力](https://learn.microsoft.com/en-us/graph/api/resources/search-api-overview?view=graph-rest-1.0)和[检索文件](https://learn.microsoft.com/en-us/graph/api/driveitem-get?view=graph-rest-1.0\&tabs=http)的能力，使 GPT Action 能够结合用户在 SharePoint 或 Office365 中可访问的文件内容来回答用户的问题。它使用 Azure Functions 处理 Graph API 的响应，将其转换为人类可读的格式，或组织成 ChatGPT 能够理解的结构。这段代码仅作为方向性参考，你应根据自己的需求进行修改。

该方案在 Azure Function 中对文件进行预处理。Azure Function 返回的是文本，而不是 base64 编码的文件。由于经过了预处理并转换为文本，该方案最适合用于大型非结构化文档，以及需要分析的文件数量超过第一种方案所支持数量的情况（请参阅[此处](https://platform.openai.com/docs/actions/sending-files)的文档）。

### 价值与业务用例示例

**价值**：用户现在可以利用 ChatGPT 的自然语言能力直接连接 Sharepoint 中的文件

**用例示例**： 
- 用户需要查找哪些文件与某个主题相关
- 用户需要找到一个深埋在文档中的关键问题的答案

## 架构 / 示例

![](../../../images/solution_2.gif)

该方案使用一个 Node.js Azure Function，基于已登录的用户执行以下操作：

1. 根据用户的初始问题，搜索用户有权访问的相关文件。

2. 对找到的每个文件，将其转换为统一的可读格式并获取全部文本。

3. 使用 GPT 4o mini（gpt-4o-mini）根据用户的初始问题从文件中提取相关文本。GPT 4o mini 的定价请参阅[此处](https://openai.com/pricing#language-models)——由于我们处理的是较小的 token 块，这一步的成本微乎其微。  

4. 将这些数据返回给 ChatGPT。随后 GPT 使用这些信息回答用户的初始问题。

从下面的架构图可以看出，前三步与方案 1 相同。主要区别在于，该方案将文件转换为文本而不是 base64 字符串，然后使用 GPT 4o mini 对文本进行摘要。

![](../../../images/solution_2_architecture.png)

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- 应用网站：https://www.microsoft.com/en-us/microsoft-365/sharepoint/collaboration
- 应用 API 文档：https://learn.microsoft.com/en-us/previous-versions/office/developer/sharepoint-rest-reference/

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 可访问 Sharepoint 环境 
- Postman（以及 API 和 OAuth 相关知识）
- 来自 platform.openai.com 的 OpenAI API 密钥

## 中间件信息

如果按照[文件搜索概念指南](https://learn.microsoft.com/en-us/graph/search-concept-files)操作，[Microsoft Graph Search API](https://learn.microsoft.com/en-us/graph/search-concept-files) 返回的是符合条件的文件引用，而不是文件内容本身。因此需要中间件，而不能直接调用 MSFT 端点。

步骤： 

1. 遍历返回的文件，并使用[下载文件端点](https://learn.microsoft.com/en-us/graph/api/driveitem-get-content?view=graph-rest-1.0\&tabs=http)或[转换文件端点](https://learn.microsoft.com/en-us/graph/api/driveitem-get-content-format?view=graph-rest-1.0\&tabs=http)下载文件

2. 使用 [pdf-parse](https://www.npmjs.com/package/pdf-parse) 将二进制流转换为人类可读的文本

3. 然后，我们可以在函数中使用 gpt-4o-mini 进行摘要来进一步优化，以应对目前 Actions 的 100,000 字符限制。 

### 其他步骤

#### 设置 Azure Function

1. 按照 [Azure Function cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_azure_function) 中的步骤设置 Azure Function

#### 添加函数代码

现在你已经有了一个经过身份验证的 Azure Function，我们可以更新该函数来搜索 SharePoint / O365

2. 打开你的测试函数，粘贴[此文件](https://github.com/openai/openai-cookbook/blob/main/examples/chatgpt/sharepoint_azure_function/solution_two_preprocessing.js)中的代码。保存该函数。

> **这段代码仅作为方向性参考**——虽然它应该可以开箱即用，但其设计初衷是让你根据自己的需求进行定制（参见本文档末尾的示例）。

3. 前往左侧 **Settings** 下的 **Configuration** 选项卡，设置以下环境变量。注意，根据你的 Azure 界面，这可能直接列在 **Environment Variables** 中。

    1. `TENANT_ID`：从上一节复制

    2. `CLIENT_ID`：从上一节复制 

    3. `OPENAI_API_KEY:` 在 platform.openai.com 上创建一个 OpenAI API 密钥。

4. 前往 **Development Tools** 下的 **Console** 选项卡

    1. 在控制台中安装以下软件包

       1. `npm install @microsoft/microsoft-graph-client`

       2. `npm install axios`

       3. `npm install pdf-parse`

       4. `npm install openai`

5. 完成后，再次尝试从 Postman 调用该函数（POST 调用），并将以下内容放入请求体中（使用你认为能产生响应的查询和搜索词）。

    ```json
    {
        "query": "<choose a question>",
        "searchTerm": "<choose a search term>"
    }
    ```

6. 如果收到了响应，就可以开始将其配置到自定义 GPT 中了！


## 详细演练

下面介绍该方案特有的设置说明和演练，即在 Azure Function 中预处理文件并提取摘要。完整代码请见[此处](https://github.com/openai/openai-cookbook/blob/main/examples/chatgpt/sharepoint_azure_function/solution_two_preprocessing.js)。

### 代码演练

#### 实现身份验证 

下面是我们将在函数中使用的几个辅助函数。

#### 初始化 Microsoft Graph 客户端
创建一个函数，使用访问令牌初始化 Graph 客户端。它将用于搜索 Office 365 和 SharePoint。

```javascript
const { Client } = require('@microsoft/microsoft-graph-client');

function initGraphClient(accessToken) {
    return Client.init({
        authProvider: (done) => {
            done(null, accessToken);
        }
    });
}
```

#### 获取 On-Behalf-Of (OBO) 令牌
该函数使用现有的 bearer 令牌向 Microsoft 身份平台请求 OBO 令牌。这样可以透传凭据，确保搜索只返回已登录用户有权访问的文件。

```javascript
const axios = require('axios');
const qs = require('querystring');

async function getOboToken(userAccessToken) {
    const { TENANT_ID, CLIENT_ID, MICROSOFT_PROVIDER_AUTHENTICATION_SECRET } = process.env;
    const params = {
        client_id: CLIENT_ID,
        client_secret: MICROSOFT_PROVIDER_AUTHENTICATION_SECRET,
        grant_type: 'urn:ietf:params:oauth:grant-type:jwt-bearer',
        assertion: userAccessToken,
        requested_token_use: 'on_behalf_of',
        scope: 'https://graph.microsoft.com/.default'
    };

    const url = `https\://login.microsoftonline.com/${TENANT_ID}/oauth2/v2.0/token`;
    try {
        const response = await axios.post(url, qs.stringify(params), {
            headers: { 'Content-Type': 'application/x-www-form-urlencoded' }
        });
        return response.data.access\_token;
    } catch (error) {
        console.error('Error obtaining OBO token:', error.response?.data || error.message);
        throw error;
    }
}
```

#### 从 O365 / SharePoint 条目中获取内容

该函数获取云盘条目的内容，处理不同的文件类型，并在需要时将文件转换为 PDF 以提取文本。对于 PDF 使用[下载端点](https://learn.microsoft.com/en-us/graph/api/driveitem-get-content?view=graph-rest-1.0\&tabs=http)，对于其他支持的文件类型则使用[转换端点](https://learn.microsoft.com/en-us/graph/api/driveitem-get-content-format?view=graph-rest-1.0\&tabs=http)。
```javascript
const getDriveItemContent = async (client, driveId, itemId, name) => {
    try {
        const fileType = path.extname(name).toLowerCase();
        // the below files types are the ones that are able to be converted to PDF to extract the text. See https://learn.microsoft.com/en-us/graph/api/driveitem-get-content-format?view=graph-rest-1.0&tabs=http
        const allowedFileTypes = ['.pdf', '.doc', '.docx', '.odp', '.ods', '.odt', '.pot', '.potm', '.potx', '.pps', '.ppsx', '.ppsxm', '.ppt', '.pptm', '.pptx', '.rtf'];
        // filePath changes based on file type, adding ?format=pdf to convert non-pdf types to pdf for text extraction, so all files in allowedFileTypes above are converted to pdf
        const filePath = `/drives/${driveId}/items/${itemId}/content` + ((fileType === '.pdf' || fileType === '.txt' || fileType === '.csv') ? '' : '?format=pdf');
        if (allowedFileTypes.includes(fileType)) {
            response = await client.api(filePath).getStream();
            // The below takes the chunks in response and combines
            let chunks = [];
            for await (let chunk of response) {
                chunks.push(chunk);
            }
            let buffer = Buffer.concat(chunks);
            // the below extracts the text from the PDF.
            const pdfContents = await pdfParse(buffer);
            return pdfContents.text;
        } else if (fileType === '.txt') {
            // If the type is txt, it does not need to create a stream and instead just grabs the content
            response = await client.api(filePath).get();
            return response;
        }  else if (fileType === '.csv') {
            response = await client.api(filePath).getStream();
            let chunks = [];
            for await (let chunk of response) {
                chunks.push(chunk);
            }
            let buffer = Buffer.concat(chunks);
            let dataString = buffer.toString('utf-8');
            return dataString
            
    } else {
        return 'Unsupported File Type';
    }
     
    } catch (error) {
        console.error('Error fetching drive content:', error);
        throw new Error(`Failed to fetch content for ${name}: ${error.message}`);
    }
};
```

#### 集成 GPT 4o mini 进行文本分析

该函数利用 OpenAI SDK 分析从文档中提取的文本，并根据用户查询找出相关信息。这有助于确保只把与用户问题相关的文本返回给 GPT。 

```javascript
const getRelevantParts = async (text, query) => {
    try {
        // We use your OpenAI key to initialize the OpenAI client
        const openAIKey = process.env["OPENAI_API_KEY"];
        const openai = new OpenAI({
            apiKey: openAIKey,
        });
        const response = await openai.chat.completions.create({
            // Using gpt-4o-mini due to speed to prevent timeouts. You can tweak this prompt as needed
            model: "gpt-4o-mini",
            messages: [
                {"role": "system", "content": "You are a helpful assistant that finds relevant content in text based on a query. You only return the relevant sentences, and you return a maximum of 10 sentences"},
                {"role": "user", "content": `Based on this question: **"${query}"**, get the relevant parts from the following text:*****\n\n${text}*****. If you cannot answer the question based on the text, respond with 'No information provided'`}
            ],
            // using temperature of 0 since we want to just extract the relevant content
            temperature: 0,
            // using max_tokens of 1000, but you can customize this based on the number of documents you are searching. 
            max_tokens: 1000
        });
        return response.choices[0].message.content;
    } catch (error) {
        console.error('Error with OpenAI:', error);
        return 'Error processing text with OpenAI' + error;
    }
};
```

#### 创建处理请求的 Azure Function

有了这些辅助函数后，Azure Function 将负责编排整个流程：对用户进行身份验证、执行搜索，并遍历搜索结果以提取文本，再将文本中的相关部分返回给 GPT。

**处理 HTTP 请求：** 函数首先从 HTTP 请求中提取 query 和 searchTerm。它会检查是否存在 Authorization 请求头，并提取 bearer 令牌。

**身份验证：** 使用 bearer 令牌，通过上面定义的 getOboToken 从 Microsoft 身份平台获取 OBO 令牌。

**初始化 Graph 客户端：** 使用 OBO 令牌，通过上面定义的 initGraphClient 初始化 Microsoft Graph 客户端。

**文档搜索：** 构造搜索查询并发送到 Microsoft Graph API，根据 searchTerm 查找文档。

**文档处理**：对于搜索返回的每个文档：

- 使用 getDriveItemContent 获取文档内容。

- 如果文件类型受支持，则使用 getRelevantParts 分析内容，该函数会将文本发送给 OpenAI 模型，根据查询提取相关信息。

- 收集分析结果，并附上文档名称和 URL 等元数据。

**响应**：函数按相关性对结果排序，并通过 HTTP 响应返回。

```javascript
module.exports = async function (context, req) {
    const query = req.query.query || (req.body && req.body.query);
    const searchTerm = req.query.searchTerm || (req.body && req.body.searchTerm);
    if (!req.headers.authorization) {
        context.res = {
            status: 400,
            body: 'Authorization header is missing'
        };
        return;
    }
    /// The below takes the token passed to the function, to use to get an OBO token.
    const bearerToken = req.headers.authorization.split(' ')[1];
    let accessToken;
    try {
        accessToken = await getOboToken(bearerToken);
    } catch (error) {
        context.res = {
            status: 500,
            body: `Failed to obtain OBO token: ${error.message}`
        };
        return;
    }
    // Initialize the Graph Client using the initGraphClient function defined above
    let client = initGraphClient(accessToken);
    // this is the search body to be used in the Microsoft Graph Search API: https://learn.microsoft.com/en-us/graph/search-concept-files
    const requestBody = {
        requests: [
            {
                entityTypes: ['driveItem'],
                query: {
                    queryString: searchTerm
                },
                from: 0,
                // the below is set to summarize the top 10 search results from the Graph API, but can configure based on your documents. 
                size: 10
            }
        ]
    };

    try { 
        // Function to tokenize content (e.g., based on words). 
        const tokenizeContent = (content) => {
            return content.split(/\s+/);
        };

        // Function to break tokens into 10k token windows for gpt-4o-mini
        const breakIntoTokenWindows = (tokens) => {
            const tokenWindows = []
            const maxWindowTokens = 10000; // 10k tokens
            let startIndex = 0;

            while (startIndex < tokens.length) {
                const window = tokens.slice(startIndex, startIndex + maxWindowTokens);
                tokenWindows.push(window);
                startIndex += maxWindowTokens;
            }

            return tokenWindows;
        };
        // This is where we are doing the search
        const list = await client.api('/search/query').post(requestBody);

        const processList = async () => {
            // This will go through and for each search response, grab the contents of the file and summarize with gpt-4o-mini
            const results = [];

            await Promise.all(list.value[0].hitsContainers.map(async (container) => {
                for (const hit of container.hits) {
                    if (hit.resource["@odata.type"] === "#microsoft.graph.driveItem") {
                        const { name, id } = hit.resource;
                        // We use the below to grab the URL of the file to include in the response
                        const webUrl = hit.resource.webUrl.replace(/\s/g, "%20");
                        // The Microsoft Graph API ranks the responses, so we use this to order it
                        const rank = hit.rank;
                        // The below is where the file lives
                        const driveId = hit.resource.parentReference.driveId;
                        const contents = await getDriveItemContent(client, driveId, id, name);
                        if (contents !== 'Unsupported File Type') {
                            // Tokenize content using function defined previously
                            const tokens = tokenizeContent(contents);

                            // Break tokens into 10k token windows
                            const tokenWindows = breakIntoTokenWindows(tokens);

                            // Process each token window and combine results
                            const relevantPartsPromises = tokenWindows.map(window => getRelevantParts(window.join(' '), query));
                            const relevantParts = await Promise.all(relevantPartsPromises);
                            const combinedResults = relevantParts.join('\n'); // Combine results

                            results.push({ name, webUrl, rank, contents: combinedResults });
                        } 
                        else {
                            results.push({ name, webUrl, rank, contents: 'Unsupported File Type' });
                        }
                    }
                }
            }));

            return results;
        };
        let results;
        if (list.value[0].hitsContainers[0].total == 0) {
            // Return no results found to the API if the Microsoft Graph API returns no results
            results = 'No results found';
        } else {
            // If the Microsoft Graph API does return results, then run processList to iterate through.
            results = await processList();
            results.sort((a, b) => a.rank - b.rank);
        }
        context.res = {
            status: 200,
            body: results
        };
    } catch (error) {
        context.res = {
            status: 500,
            body: `Error performing search or processing results: ${error.message}`,
        };
    }
};
```

### 定制

以下是一些可以定制的方向。 

- 你可以定制 GPT 提示词，使其在没有找到结果时再搜索一定次数。

- 你可以通过定制搜索查询，让代码只搜索特定的 SharePoint 站点或 O365 云盘。这有助于聚焦搜索并改善检索效果。按当前设置，该函数会搜索已登录用户可访问的所有文件。

- 你可以使用 gpt-4o 代替 gpt-4o-mini。这会略微增加成本和延迟，但可能得到质量更高的摘要。

- 你可以在调用 Microsoft Graph 时定制搜索的文件数量。


### 注意事项

请注意，Actions 的所有限制在这里同样适用，包括返回内容不超过 10 万字符以及 [45 秒超时](https://platform.openai.com/docs/actions/production/timeouts)。


- 这只适用于文本，不适用于图像。通过在 Azure Function 中添加一些额外代码，你可以使用 GPT-4o 提取图像摘要来实现这一点。

- 这不适用于结构化数据。如果结构化数据是你用例的主要部分，我们推荐使用方案 1。

## ChatGPT 步骤

### 自定义 GPT 指令 

创建自定义 GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

In [ ]:
You are a Q&A helper that helps answer users questions. You have access to a documents repository through your API action. When a user asks a question, you pass in that question exactly as stated to the "query" parameter, and for the "searchTerm" you use a single keyword or term you think you should use for the search.

****

Scenario 1: There are answers

If your action returns results, then you take the results from the action and summarize concisely with the webUrl returned from the action. You answer the users question to the best of your knowledge from the action

****

Scenario 2: No results found

If the response you get from the action is "No results found", stop there and let the user know there were no results and that you are going to try a different search term, and explain why. You must always let the user know before conducting another search.

Example:

****

I found no results for "DEI". I am now going to try [insert term] because [insert explanation]

****

Then, try a different searchTerm that is similar to the one you tried before, with a single word. 

Try this three times. After the third time, then let the user know you did not find any relevant documents to answer the question, and to check SharePoint. Be sure to be explicit about what you are searching for at each step.

****

In either scenario, try to answer the user's question. If you cannot answer the user's question based on the knowledge you find, let the user know and ask them to go check the HR Docs in SharePoint. If the file is a CSV, XLSX, or XLS, you can tell the user to download the file using the link and re-upload to use Advanced Data Analysis.

### OpenAPI Schema 

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

下面的规范传入 `query` 参数用于指导预处理，并传入 `searchTerm` 用于在 Microsoft Graph 中查找正确的文件。
>请务必根据上方截图中复制的链接，替换函数应用名称、函数名称和 code

In [ ]:
openapi: 3.1.0
info:
  title: SharePoint Search API
  description: API for searching SharePoint documents.
  version: 1.0.0
servers:
  - url: https://{your_function_app_name}.azurewebsites.net/api
    description: SharePoint Search API server
paths:
  /{your_function_name}?code={enter your specific endpoint id here}:
    post:
      operationId: searchSharePoint
      summary: Searches SharePoint for documents matching a query and term.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                query:
                  type: string
                  description: The full query to search for in SharePoint documents.
                searchTerm:
                  type: string
                  description: A specific term to search for within the documents.
      responses:
        '200':
          description: Search results
          content:
            application/json:
              schema:
                type: array
                items:
                  type: object
                  properties:
                    documentName:
                      type: string
                      description: The name of the document.
                    snippet:
                      type: string
                      description: A snippet from the document containing the search term.
                    url:
                      type: string
                      description: The URL to access the document.

## 身份验证说明

以下是与该第三方应用设置身份验证的说明。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

*有关身份验证的更详细说明，请参阅上文以及 [Azure Function cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_azure_function)。*

## 常见问题与故障排除

- 为什么代码中使用 Microsoft Graph API，而不是 [SharePoint API](https://learn.microsoft.com/en-us/sharepoint/dev/sp-add-ins/get-to-know-the-sharepoint-rest-service?tabs=csom)？

  - SharePoint API 已是旧版——根据[此处](https://learn.microsoft.com/en-us/sharepoint/dev/apis/sharepoint-rest-graph)的 Microsoft 文档，“对于 SharePoint Online，针对 SharePoint 使用 REST API 的创新是通过 Microsoft Graph REST API 推动的。”Graph API 为我们提供了更大的灵活性，而 SharePoint API 仍会遇到[为什么需要这样做，而不是直接与 Microsoft Graph API 交互？](#why-is-this-necessary-instead-of-interacting-with-the-microsoft-api-directly)一节中列出的相同文件问题。

- 支持哪些类型的文件？
    1. 支持[_此处_](https://learn.microsoft.com/en-us/graph/api/driveitem-get-content-format?view=graph-rest-1.0\&tabs=http)转换文件端点文档中列出的所有文件。具体来说，支持 _pdf, doc, docx, odp, ods, odt, pot, potm, potx, pps, ppsx, ppsxm, ppt, pptm, pptx, rtf_。

    2. 当搜索结果返回 XLS、XLSX 或 CSV 时，会提示用户下载文件并重新上传，以便使用高级数据分析提问。如上所述，如果结构化数据是你用例的一部分，我们推荐使用方案 1。

- 为什么需要请求 OBO 令牌？

  - 如果你尝试用登录 Azure Function 时使用的同一个令牌来向 Graph API 进行身份验证，会得到“invalid audience”（受众无效）错误。这是因为该令牌的受众只能是 user\_impersonation。

  - 为了解决这个问题，该函数使用 [On Behalf Of 流程](https://learn.microsoft.com/en-us/entra/identity-platform/v2-oauth2-on-behalf-of-flow)在应用内请求一个作用域为 Files.Read.All 的新令牌。它会继承已登录用户的权限，这意味着该函数只会搜索已登录用户有权访问的文件。 

  - 我们有意在每次请求时都请求一个新的 On Behalf Of 令牌，因为 Azure Function 应用本应是无状态的。你也可以将其与 Azure Key Vault 集成，用于存储密钥并以编程方式获取。 

*有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 github 上提交 PR 或 issue，我们会查看。*
